# Compositional Generalization Breakthrough Experiment Suite
## Investigating Training Dynamics, Grokking, and Cumulative Exposure

This self-contained Kaggle notebook executes two decisive experiments:
1. **Experiment 1: The 50,000-Step Grokking Test** — Tests whether standard baseline training remains permanently trapped at ~45.9% OOD (asymptotically stable attractor) or spontaneously groks at 20k–50k steps under weight decay.
2. **Experiment 2: Equal Cumulative Exposure Matrix** — Holds total integrated compositional supervision strictly constant ($\int_0^T \lambda(t) dt = 400$) across 4 temporal distributions: **Constant Low** ($\lambda=0.2$), **Early Burst** ($\lambda=1.0$ at steps 0–400), **Late Burst** ($\lambda=1.0$ at steps 1600–2000), and **Pulsed** ($\lambda=1.0$ every 5 steps).

### Output Files (Saved to `/kaggle/working/breakthrough_output/`):
- `grokking_results.pkl` and `grokking_trajectories.csv`
- `exposure_results.pkl` and `exposure_trajectories.csv`
- `figure_grokking_50k.png` and `figure_exposure_matrix.png`

In [ ]:
# =============================================================================
# CELL 1: Environment Setup, Reproducibility, and GPU Verification
# =============================================================================
import os
import pickle
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

OUT_DIR = "/kaggle/working/breakthrough_output" if os.path.isdir("/kaggle") else "./output/breakthrough"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()

print(f"Device: {device} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


In [ ]:
# =============================================================================
# CELL 2: Self-Contained H-Bar Dataset Generator & Vocabulary
# =============================================================================
class HBarVocab:
    def __init__(self):
        self.pad_token = "<pad>"
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.primitives = ["jump", "walk", "run", "look"]
        self.modifiers = ["twice", "thrice"]
        self.combinators = ["and", "after", "around", "opposite"]

        self.src_tokens = [self.pad_token, self.bos_token, self.eos_token] + self.primitives + self.modifiers + self.combinators
        self.tgt_tokens = [self.pad_token, self.bos_token, self.eos_token] + [
            f"{act}_{mod}" for act in ["I_JUMP", "I_WALK", "I_RUN", "I_LOOK"] for mod in ["NONE", "TWICE", "THRICE", "AROUND", "OPPOSITE"]
        ] + ["I_TURN_LEFT", "I_TURN_RIGHT", "I_AND", "I_AFTER"]

        self.src2idx = {t: i for i, t in enumerate(self.src_tokens)}
        self.idx2src = {i: t for i, t in enumerate(self.src_tokens)}
        self.tgt2idx = {t: i for i, t in enumerate(self.tgt_tokens)}
        self.idx2tgt = {i: t for i, t in enumerate(self.tgt_tokens)}

    def __len__(self):
        return max(len(self.src_tokens), len(self.tgt_tokens))

def interpret_hbar_command(cmd_tokens):
    out = []
    i = 0
    while i < len(cmd_tokens):
        token = cmd_tokens[i]
        if token in ["jump", "walk", "run", "look"]:
            action = f"I_{token.upper()}"
            if i + 1 < len(cmd_tokens) and cmd_tokens[i+1] in ["twice", "thrice"]:
                mod = cmd_tokens[i+1].upper()
                out.append(f"{action}_{mod}")
                i += 2
            elif i + 1 < len(cmd_tokens) and cmd_tokens[i+1] in ["around", "opposite"]:
                mod = cmd_tokens[i+1].upper()
                out.append(f"{action}_{mod}")
                i += 2
            else:
                out.append(f"{action}_NONE")
                i += 1
        elif token in ["and", "after"]:
            out.append(f"I_{token.upper()}")
            i += 1
        else:
            i += 1
    return out

class HBarDataset(Dataset):
    def __init__(self, pairs, vocab, max_len=16):
        self.pairs = pairs
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_tokens, tgt_tokens = self.pairs[idx]
        src_ids = [self.vocab.src2idx.get(t, 0) for t in src_tokens][:self.max_len]
        tgt_ids = [self.vocab.tgt2idx["<bos>"]] + [self.vocab.tgt2idx.get(t, 0) for t in tgt_tokens][:self.max_len-2] + [self.vocab.tgt2idx["<eos>"]]

        src_pad = src_ids + [0] * (self.max_len - len(src_ids))
        tgt_pad = tgt_ids + [0] * (self.max_len - len(tgt_ids))

        return torch.tensor(src_pad, dtype=torch.long), torch.tensor(tgt_pad, dtype=torch.long)

def generate_hbar_splits(seed=42):
    random.seed(seed)
    vocab = HBarVocab()
    prims = vocab.primitives
    mods = vocab.modifiers
    combs = vocab.combinators

    train_cmds = []
    for p in prims:
        train_cmds.append([p])
        for m in mods:
            train_cmds.append([p, m])

    for p1 in prims:
        for c in ["and", "after"]:
            for p2 in prims:
                train_cmds.append([p1, c, p2])
                for m in mods:
                    train_cmds.append([p1, m, c, p2])
                    train_cmds.append([p1, c, p2, m])

    train_pool = []
    while len(train_pool) < 16000:
        cmd = random.choice(train_cmds)
        train_pool.append((cmd, interpret_hbar_command(cmd)))

    id_test = []
    while len(id_test) < 4000:
        cmd = random.choice(train_cmds)
        id_test.append((cmd, interpret_hbar_command(cmd)))

    ood_test = []
    ood_templates = []
    for p1 in prims:
        for p2 in prims:
            for p3 in prims:
                for c1 in combs:
                    for c2 in combs:
                        ood_templates.append([p1, c1, p2, "twice", c2, p3, "thrice"])
                        ood_templates.append([p1, "around", c1, p2, "opposite", c2, p3])

    while len(ood_test) < 1500:
        cmd = random.choice(ood_templates)
        ood_test.append((cmd, interpret_hbar_command(cmd)))

    comp_pool = []
    for cmd, _ in train_pool[:8000]:
        swapped = [random.choice(prims) if t in prims else t for t in cmd]
        comp_pool.append((swapped, interpret_hbar_command(swapped)))

    return train_pool, id_test, ood_test, comp_pool, vocab

print("Generating H-Bar dataset splits...")
train_pairs, id_pairs, ood_pairs, comp_pairs, vocab = generate_hbar_splits(seed=42)
print(f"Dataset generated: Train={len(train_pairs)} | ID={len(id_pairs)} | OOD={len(ood_pairs)} | Comp={len(comp_pairs)}")


In [ ]:
# =============================================================================
# CELL 3: 2-Layer Transformer Architecture
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class HBarTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src_mask = (src == self.pad_idx)
        tgt_mask = (tgt == self.pad_idx)
        seq_len = tgt.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=src.device).bool()

        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=causal_mask,
            src_key_padding_mask=src_mask,
            tgt_key_padding_mask=tgt_mask
        )
        return self.fc_out(out)

def evaluate_accuracy(model, loader, device, max_batches=8):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for i, (src, tgt) in enumerate(loader):
            if i >= max_batches:
                break
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt[:, :-1])
            preds = out.argmax(dim=-1)
            mask = tgt[:, 1:] != 0
            correct += ((preds == tgt[:, 1:]) & mask).sum().item()
            total += mask.sum().item()
    model.train()
    return 100.0 * correct / total if total > 0 else 0.0


In [ ]:
# =============================================================================
# CELL 4: High-Throughput Training Loop with Real-Time Progress Output
# =============================================================================
def train_model(
    condition_name,
    mode="baseline",
    comp_weight=0.0,
    comp_every=1,
    start_step=0,
    end_step=2000,
    weight_decay=0.0,
    n_timesteps=2000,
    eval_every=100,
    seed=42,
    batch_size=64,
    lr=1e-3
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    train_ds = HBarDataset(train_pairs, vocab)
    id_ds = HBarDataset(id_pairs, vocab)
    ood_ds = HBarDataset(ood_pairs, vocab)
    comp_ds = HBarDataset(comp_pairs, vocab)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    id_loader = DataLoader(id_ds, batch_size=batch_size, shuffle=False)
    ood_loader = DataLoader(ood_ds, batch_size=batch_size, shuffle=False)
    comp_loader = DataLoader(comp_ds, batch_size=batch_size, shuffle=True)

    model = HBarTransformer(vocab_size=len(vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scaler = GradScaler("cuda", enabled=use_amp)

    history = {"step": [], "loss": [], "acc_id": [], "acc_ood": [], "comp_weight": []}

    train_iter = iter(train_loader)
    comp_iter = iter(comp_loader)

    pbar = tqdm(total=n_timesteps, desc=f"[{condition_name}]", unit="step")
    t_start = time.time()

    for step in range(n_timesteps):
        try:
            src, tgt = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            src, tgt = next(train_iter)

        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            out = model(src, tgt[:, :-1])
            task_loss = criterion(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))

        curr_weight = 0.0
        if mode == "constant":
            if step % comp_every == 0:
                curr_weight = comp_weight
        elif mode == "window":
            if start_step <= step < end_step and step % comp_every == 0:
                curr_weight = comp_weight
        elif mode == "pulsed":
            if step % comp_every == 0:
                curr_weight = comp_weight

        total_loss = task_loss
        if curr_weight > 0.0:
            try:
                c_src, c_tgt = next(comp_iter)
            except StopIteration:
                comp_iter = iter(comp_loader)
                c_src, c_tgt = next(comp_iter)
            c_src, c_tgt = c_src.to(device), c_tgt.to(device)
            with autocast("cuda", enabled=use_amp):
                c_out = model(c_src, c_tgt[:, :-1])
                c_loss = criterion(c_out.reshape(-1, c_out.size(-1)), c_tgt[:, 1:].reshape(-1))
            total_loss = total_loss + curr_weight * c_loss

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if step % eval_every == 0 or step == n_timesteps - 1:
            acc_id = evaluate_accuracy(model, id_loader, device)
            acc_ood = evaluate_accuracy(model, ood_loader, device)
            history["step"].append(step)
            history["loss"].append(float(task_loss.item()))
            history["acc_id"].append(acc_id)
            history["acc_ood"].append(acc_ood)
            history["comp_weight"].append(curr_weight)

            pbar.set_postfix({
                "Loss": f"{task_loss.item():.3f}",
                "ID": f"{acc_id:.1f}%",
                "OOD": f"{acc_ood:.1f}%",
                "Step": f"{step}/{n_timesteps}"
            })

        pbar.update(1)

    pbar.close()
    print(f"Done: {condition_name} ({time.time()-t_start:.1f}s) | Final ID: {history['acc_id'][-1]:.1f}% | Final OOD: {history['acc_ood'][-1]:.1f}%")
    return history


In [ ]:
# =============================================================================
# CELL 5: EXPERIMENT 1 — 20,000-Step Grokking vs Asymptotic Trap Test
# =============================================================================
print("=" * 70)
print("RUNNING EXPERIMENT 1: EXTENDED GROKKING TEST (BASELINE vs WEIGHT DECAY)")
print("=" * 70)

# 20,000 steps per arm (takes ~2.5 mins per arm on Kaggle GPU!)
n_grok_steps = 20000
eval_grok_every = 200

grokking_runs = {}
conditions_grok = [
    ("Baseline (No WD)", 0.0),
    ("Baseline (WD=0.01)", 0.01),
    ("Baseline (WD=0.10)", 0.10),
]

for name, wd in conditions_grok:
    hist = train_model(
        condition_name=name,
        mode="baseline",
        weight_decay=wd,
        n_timesteps=n_grok_steps,
        eval_every=eval_grok_every,
        seed=42
    )
    grokking_runs[name] = hist

# Plot Experiment 1
plt.figure(figsize=(10, 5))
for name, hist in grokking_runs.items():
    plt.plot(hist["step"], hist["acc_ood"], label=f"{name} (Final: {hist['acc_ood'][-1]:.1f}%)", linewidth=2)

plt.axhline(45.9, color="black", linestyle="--", alpha=0.7, label="2,000-step baseline plateau (45.9%)")
plt.title(f"{n_grok_steps}-Step Grokking Test: Does Baseline Spontaneously Generalize?", fontsize=13, fontweight="bold")
plt.xlabel("Training Step", fontsize=11)
plt.ylabel("OOD Accuracy (%)", fontsize=11)
plt.ylim(0, 105)
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figure_grokking_50k.png", dpi=300)
plt.show()


In [ ]:
# =============================================================================
# CELL 6: EXPERIMENT 2 — Equal Cumulative Exposure Matrix (Early vs Late vs Constant vs Pulsed)
# =============================================================================
print("=" * 70)
print("RUNNING EXPERIMENT 2: EQUAL CUMULATIVE EXPOSURE MATRIX (Total Integral = 400)")
print("=" * 70)

exposure_configs = {
    "Constant Low (λ=0.2)": {"mode": "constant", "comp_weight": 0.2, "comp_every": 1},
    "Early Burst (λ=1.0, 0-400)": {"mode": "window", "comp_weight": 1.0, "comp_every": 1, "start_step": 0, "end_step": 400},
    "Late Burst (λ=1.0, 1600-2000)": {"mode": "window", "comp_weight": 1.0, "comp_every": 1, "start_step": 1600, "end_step": 2000},
    "Pulsed (λ=1.0, every 5th)": {"mode": "pulsed", "comp_weight": 1.0, "comp_every": 5},
}

exposure_runs = {}
for name, p in exposure_configs.items():
    hist = train_model(
        condition_name=name,
        mode=p["mode"],
        comp_weight=p["comp_weight"],
        comp_every=p.get("comp_every", 1),
        start_step=p.get("start_step", 0),
        end_step=p.get("end_step", 2000),
        n_timesteps=2000,
        eval_every=25,
        seed=42
    )
    exposure_runs[name] = hist

# Plot Experiment 2
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True, gridspec_kw={'height_ratios': [2.5, 1]})

for name, hist in exposure_runs.items():
    ax1.plot(hist["step"], hist["acc_ood"], label=f"{name} (Final: {hist['acc_ood'][-1]:.1f}%)", linewidth=2)
    ax2.plot(hist["step"], hist["comp_weight"], label=name, alpha=0.8)

ax1.set_title("Equal Cumulative Exposure Matrix: Timing Distribution vs Recovery", fontsize=13, fontweight="bold")
ax1.set_ylabel("OOD Accuracy (%)", fontsize=11)
ax1.set_ylim(0, 105)
ax1.grid(True, alpha=0.3)
ax1.legend(loc="lower right", fontsize=10)

ax2.set_title("Applied Compositional Loss Weight λ(t) [All Sum to 400 Total Units]", fontsize=11)
ax2.set_xlabel("Training Step", fontsize=11)
ax2.set_ylabel("λ(t)", fontsize=11)
ax2.set_ylim(-0.05, 1.15)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figure_exposure_matrix.png", dpi=300)
plt.show()


In [ ]:
# =============================================================================
# CELL 7: Export Results to CSV and Pickle for Manuscript Integration
# =============================================================================
with open(f"{OUT_DIR}/grokking_results.pkl", "wb") as f:
    pickle.dump(grokking_runs, f)
with open(f"{OUT_DIR}/exposure_results.pkl", "wb") as f:
    pickle.dump(exposure_runs, f)

# Convert to long-form DataFrames
grok_dfs = []
for name, hist in grokking_runs.items():
    df = pd.DataFrame(hist)
    df["condition"] = name
    grok_dfs.append(df)
pd.concat(grok_dfs).to_csv(f"{OUT_DIR}/grokking_trajectories.csv", index=False)

exp_dfs = []
for name, hist in exposure_runs.items():
    df = pd.DataFrame(hist)
    df["condition"] = name
    exp_dfs.append(df)
pd.concat(exp_dfs).to_csv(f"{OUT_DIR}/exposure_trajectories.csv", index=False)

print(f"\nAll results and high-resolution figures successfully exported to: {OUT_DIR}/")
